In [2]:
import pandas as pd
import numpy as np

# Load the CDC data - check all sheets first
excel_file = pd.ExcelFile('../data/raw/cdc_251644_DS1.xlsx')
print("Available sheets:")
for i, sheet in enumerate(excel_file.sheet_names):
    print(f"{i}: {sheet}")

Available sheets:
0: READ ME
1: Profile of Acute Care Hospitals
2: Table of Contents
3: Table 1a-CLABSI
4: Table 1b-CAUTI
5: Table 1c-VAE 
6: Table 1d-COLO
7: Table 1d-HYST
8: Table 1e-MRSA
9: Table 1f-CDI
10: Table 1g Footnotes
11: Table 2a-NAT'L DA Data 
12: Table 2a-i-NAT'L DA Data 
13: Table 2a-ii-NAT'L DA Data
14: Table 2a-iii-NAT'L DA Data  
15: Table 2b-NAT'L LABID Data
16: Table 2c-NAT'L SSI Data
17: Table 2d-NAT'L SSI Data
18: Table 3a-State CLABSI Data
19: Table 3b-State CLABSI Data
20: Table 3c-State CLABSI Data
21: Table 3d-State CLABSI Data
22: Table 4a-State CAUTI Data
23: Table 4b-State CAUTI Data
24: Table 4c-State CAUTI Data
25: Table 5a-State VAE Data 
26: Table 5b-State VAE Data 
27: Table 5c-State VAE Data 
28: Table 6a-State SSI Data
29: Table 6b-State SSI Data
30: Table 6c-State SSI Data
31: Table 6d-State SSI Data
32: Table 6e-State SSI Data
33: Table 6f-State SSI Data
34: Table 6g-State SSI Data
35: Table 6h-State SSI Data
36: Table 6i-State SSI Data
37: Table 6

In [3]:
# Load state-level SIR comparison data 
# SIR > 1 means worse than national average, < 1 means better

# Read multiple state SIR sheets and combine
sir_data = []

for sheet_num in range(46, 53):  # Tables 10a through 10g
    sheet_name = excel_file.sheet_names[sheet_num]
    print(f"\nLoading {sheet_name}...")
    
    df = pd.read_excel('../data/raw/cdc_251644_DS1.xlsx', sheet_name=sheet_name, header=1)
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()[:5]}")  # First 5 columns
    print(df.head(3))


Loading Table 10a-State SIR Comparison...
Shape: (65, 6)
Columns: ['10a. Central line-associated bloodstream infections (CLABSI), all locations1', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
  10a. Central line-associated bloodstream infections (CLABSI), all locations1  \
0                                                NaN                             
1                                             State2                             
2                                            Alabama                             

                                     Unnamed: 1 Unnamed: 2       Unnamed: 3  \
0    All Acute Care Hospitals Reporting to NHSN        NaN              NaN   
1                                      2022 SIR   2023 SIR  Percent Change3   
2                                         1.035      0.919             0.11   

                                          Unnamed: 4 Unnamed: 5  
0                                                NaN        NaN  
1  Direction of Ch

In [4]:
# Column 0 = State names
# Column 1 = 2022 SIR 
# Column 2 = 2023 SIR 
# Column 3 = % Change
# Column 4 = Direction
# Column 5 = p value

# Rename columns to something readable
df_clabsi.columns = ['State', 'SIR_2022', 'SIR_2023', 'Percent_Change', 'Direction', 'P_Value']

print("New column names:")
print(df_clabsi.columns.tolist())
print("\nFirst 5 rows:")
print(df_clabsi.head())

NameError: name 'df_clabsi' is not defined

In [ ]:
# fix row 0
df_clabsi = df_clabsi.iloc[1:]

print("After removing header row:")
print(df_clabsi.head())

In [ ]:
# convert SIR_2023 values from txt to number 
df_clabsi['SIR_2023'] = pd.to_numeric(df_clabsi['SIR_2023'], errors='coerce')

print("Data types:")
print(df_clabsi.dtypes)
print("\nFirst 5 rows:")
print(df_clabsi.head())
print("\nAny missing values?")
print(df_clabsi.isnull().sum())

In [ ]:
# I only need State and SIR_2023 for the dashboard
df_clabsi_clean = df_clabsi[['State', 'SIR_2023']].copy()

# Remove rows where SIR_2023 is missing
df_clabsi_clean = df_clabsi_clean.dropna(subset=['SIR_2023'])

# Rename the SIR_2023 column to include the infection name
df_clabsi_clean.columns = ['State', 'CLABSI_SIR_2023']

print("Clean CLABSI data:")
print(df_clabsi_clean.head(10))
print(f"\nShape: {df_clabsi_clean.shape}")

In [9]:
# Define a function that cleans infection data

def extract_infection_data(sheet_name, infection_name):
    """
    Load and clean one infection type from CDC data.
    Returns: DataFrame with State and SIR_2023 columns
    """
    
    # Read the Excel sheet (skip first 2 rows of headers)
    df = pd.read_excel(
        '../data/raw/cdc_251644_DS1.xlsx',
        sheet_name=sheet_name,
        skiprows=2
    )
    
    # Rename columns
    df.columns = ['State', 'SIR_2022', 'SIR_2023', 'Percent_Change', 'Direction', 'P_Value']
    df['State'] = df['State'].astype(str).str.strip().replace({'D. C.': 'D.C.'})
    
    # Remove header row (row 0)
    df = df.iloc[1:]
    
    # Convert SIR_2023 to numbers
    df['SIR_2023'] = pd.to_numeric(df['SIR_2023'], errors='coerce')
    
    # Keep only State and SIR_2023
    df_clean = df[['State', 'SIR_2023']].copy()
    
    # Remove missing values
    df_clean = df_clean.dropna(subset=['SIR_2023'])
    
    # Rename SIR_2023 to include infection name
    df_clean.columns = ['State', f'{infection_name}_SIR_2023']
    
    return df_clean

# Test the function on CAUTI
df_cauti = extract_infection_data('Table 10b-State SIR Comparison', 'CAUTI')
print("CAUTI data:")
print(df_cauti.head())

CAUTI data:
        State  CAUTI_SIR_2023
1     Alabama           0.629
2      Alaska           0.909
3     Arizona           0.405
4    Arkansas           0.440
5  California           0.715


In [12]:
infections = [
    ('Table 10a-State SIR Comparison', 'CLABSI'),
    ('Table 10b-State SIR Comparison', 'CAUTI'),
    ('Table 10c-State SIR Comparison ', 'VAE'),
    ('Table 10d-State SIR Comparison', 'SSI_COLON'),
    ('Table 10e-State SIR Comparison', 'SSI_HYST'),
    ('Table 10f-State SIR Comparison', 'MRSA'),
    ('Table 10g-State SIR Comparison', 'CDI'),
]
df_all = None

for sheet_name, infection_name in infections:
    print(f"Loading {infection_name}...")
    
    df_infection = extract_infection_data(sheet_name, infection_name)
    
    # Merge with the growing master dataframe
    if df_all is None:
        df_all = df_infection
    else:
        # Merge on State column 
        df_all = df_all.merge(df_infection, on='State', how='outer')
    
    print(f"  {infection_name}: {len(df_infection)} states")

import os
os.makedirs('../data/processed', exist_ok=True)
df_all.to_csv('../data/processed/hai_state_data_2023.csv', index=False)

print(f"\nFinal dataset shape: {df_all.shape}")
print(f"States: {len(df_all)}")
print(f"Columns: {df_all.columns.tolist()}")
print("\nFirst 10 rows:")
print(df_all.head(10))

Loading CLABSI...
  CLABSI: 53 states
Loading CAUTI...
  CAUTI: 53 states
Loading VAE...
  VAE: 48 states
Loading SSI_COLON...
  SSI_COLON: 52 states
Loading SSI_HYST...
  SSI_HYST: 53 states
Loading MRSA...
  MRSA: 53 states
Loading CDI...
  CDI: 53 states

Final dataset shape: (53, 8)
States: 53
Columns: ['State', 'CLABSI_SIR_2023', 'CAUTI_SIR_2023', 'VAE_SIR_2023', 'SSI_COLON_SIR_2023', 'SSI_HYST_SIR_2023', 'MRSA_SIR_2023', 'CDI_SIR_2023']

First 10 rows:
         State  CLABSI_SIR_2023  CAUTI_SIR_2023  VAE_SIR_2023  \
0      Alabama            0.919           0.629         1.025   
1       Alaska            0.512           0.909         1.814   
2       All US            0.724           0.621         1.131   
3      Arizona            0.677           0.405         0.797   
4     Arkansas            0.667           0.440         1.947   
5   California            0.751           0.715         1.182   
6     Colorado            0.565           0.527         1.215   
7  Connecticut   

In [13]:
df_all[df_all['State'] == 'D.C.']

,State,CLABSI_SIR_2023,CAUTI_SIR_2023,VAE_SIR_2023,SSI_COLON_SIR_2023,SSI_HYST_SIR_2023,MRSA_SIR_2023,CDI_SIR_2023
8,D.C.,0.729,0.389,NaN,0.57,1.133,0.632,0.483


In [22]:
import pandas as pd

# Load the cleaned data 
df_all = pd.read_csv('../data/processed/hai_state_data_2023.csv')
df_all = df_all[df_all['State'] != 'All US'].copy()

print(f"Data loaded: {df_all.shape}")
print(df_all.head())

Data loaded: (52, 8)
        State  CLABSI_SIR_2023  CAUTI_SIR_2023  VAE_SIR_2023  \
0     Alabama            0.919           0.629         1.025   
1      Alaska            0.512           0.909         1.814   
3     Arizona            0.677           0.405         0.797   
4    Arkansas            0.667           0.440         1.947   
5  California            0.751           0.715         1.182   

   SSI_COLON_SIR_2023  SSI_HYST_SIR_2023  MRSA_SIR_2023  CDI_SIR_2023  
0               0.732              0.970          1.022         0.473  
1               1.212              1.434          0.232         0.398  
3               0.825              1.266          0.719         0.463  
4               1.146              1.103          0.919         0.363  
5               0.871              0.818          0.713         0.492  


In [23]:
# What's the average SIR for each infection type nationally?
print("AVERAGE SIR BY INFECTION TYPE (2023):")

infections_cols = [
    'CLABSI_SIR_2023',
    'CAUTI_SIR_2023', 
    'VAE_SIR_2023',
    'SSI_COLON_SIR_2023',
    'SSI_HYST_SIR_2023',
    'MRSA_SIR_2023',
    'CDI_SIR_2023'
]
national_avg = df_all[infections_cols].mean()

# Sort from worst to best
national_avg_sorted = national_avg.sort_values(ascending=False)

print(national_avg_sorted)
print("INTERPRETATION:")
print("SIR < 1.0 = Better than national average")
print("SIR > 1.0 = Worse than national average")

AVERAGE SIR BY INFECTION TYPE (2023):
VAE_SIR_2023          1.203915
SSI_HYST_SIR_2023     1.105808
SSI_COLON_SIR_2023    0.904059
CLABSI_SIR_2023       0.738615
MRSA_SIR_2023         0.691442
CAUTI_SIR_2023        0.656115
CDI_SIR_2023          0.446019
dtype: float64
INTERPRETATION:
SIR < 1.0 = Better than national average
SIR > 1.0 = Worse than national average


In [24]:
# Which states are best at preventing CLABSI?
print("BEST 5 STATES FOR CLABSI:")
print(df_all.nsmallest(5, 'CLABSI_SIR_2023')[['State', 'CLABSI_SIR_2023']])

print("\nWORST 5 STATES FOR CLABSI:")
print(df_all.nlargest(5, 'CLABSI_SIR_2023')[['State', 'CLABSI_SIR_2023']])

print("\nBEST 5 STATES FOR VAE (biggest problem):")
print(df_all.nsmallest(5, 'VAE_SIR_2023')[['State', 'VAE_SIR_2023']])

print("\nWORST 5 STATES FOR VAE:")
print(df_all.nlargest(5, 'VAE_SIR_2023')[['State', 'VAE_SIR_2023']])

BEST 5 STATES FOR CLABSI:
           State  CLABSI_SIR_2023
27       Montana            0.347
13         Idaho            0.503
52       Wyoming            0.504
1         Alaska            0.512
35  North Dakota            0.523

WORST 5 STATES FOR CLABSI:
            State  CLABSI_SIR_2023
40    Puerto Rico            2.721
50  West Virginia            0.963
0         Alabama            0.919
12         Hawaii            0.891
20          Maine            0.874

BEST 5 STATES FOR VAE (biggest problem):
            State  VAE_SIR_2023
50  West Virginia         0.433
7     Connecticut         0.677
44      Tennessee         0.730
11        Georgia         0.760
3         Arizona         0.797

WORST 5 STATES FOR VAE:
         State  VAE_SIR_2023
13       Idaho         3.058
27     Montana         2.197
4     Arkansas         1.947
32  New Mexico         1.884
1       Alaska         1.814


In [25]:
# Create an "overall score" - average SIR across all 7 infections
df_all['Overall_Score'] = df_all[infections_cols].mean(axis=1)


print("BEST 10 STATES OVERALL (lowest average SIR):")
print(df_all.nsmallest(10, 'Overall_Score')[['State', 'Overall_Score']])

print("\nWORST 10 STATES OVERALL (highest average SIR):")
print(df_all.nlargest(10, 'Overall_Score')[['State', 'Overall_Score']])

print("\nNATIONAL AVERAGE OVERALL SCORE:")
print(f"{df_all['Overall_Score'].mean():.3f}")

print("\nSTANDARD DEVIATION:")
print(f"{df_all['Overall_Score'].std():.3f}")

BEST 10 STATES OVERALL (lowest average SIR):
             State  Overall_Score
7      Connecticut       0.622857
52         Wyoming       0.655000
8             D.C.       0.656000
9         Delaware       0.671833
41    Rhode Island       0.693714
43    South Dakota       0.698143
42  South Carolina       0.698286
46            Utah       0.698714
31      New Jersey       0.701000
48        Virginia       0.712857

WORST 10 STATES OVERALL (highest average SIR):
           State  Overall_Score
40   Puerto Rico       1.168667
35  North Dakota       1.036333
12        Hawaii       1.010143
13         Idaho       0.996571
47       Vermont       0.943333
4       Arkansas       0.940714
23      Michigan       0.934143
1         Alaska       0.930143
25   Mississippi       0.917857
49    Washington       0.913000

NATIONAL AVERAGE OVERALL SCORE:
0.816

STANDARD DEVIATION:
0.108


In [26]:
# Add regions to the data
def assign_region(state):
    regions = {
        'Northeast': ['Connecticut', 'Maine', 'Massachusetts', 'New Hampshire', 'Rhode Island', 
                      'Vermont', 'New Jersey', 'New York', 'Pennsylvania'],
        'Midwest': ['Illinois', 'Indiana', 'Michigan', 'Ohio', 'Wisconsin', 'Iowa', 'Kansas', 
                    'Minnesota', 'Missouri', 'Nebraska', 'North Dakota', 'South Dakota'],
        'South': ['Delaware', 'Florida', 'Georgia', 'Maryland', 'North Carolina', 'South Carolina',
                  'Virginia', 'West Virginia', 'Alabama', 'Kentucky', 'Mississippi', 'Tennessee',
                  'Arkansas', 'Louisiana', 'Oklahoma', 'Texas'],
        'West': ['Arizona', 'Colorado', 'Idaho', 'Montana', 'Nevada', 'New Mexico', 'Utah', 
                 'Wyoming', 'Alaska', 'California', 'Hawaii', 'Oregon', 'Washington'],
    }
    for region, states in regions.items():
        if state in states:
            return region
    return 'Other'

df_all['Region'] = df_all['State'].apply(assign_region)

print("AVERAGE INFECTION RATE BY REGION:")
regional_avg = df_all.groupby('Region')['Overall_Score'].mean().sort_values()
print(regional_avg)

print("\nREGION PERFORMANCE:")
for region, score in regional_avg.items():
    if score < 0.820:
        status = "BETTER than national avg"
    else:
        status = "WORSE than national avg"
    print(f"{region:15} {score:.3f}  ({status})")

AVERAGE INFECTION RATE BY REGION:
Region
Northeast    0.784005
South        0.810124
Midwest      0.815730
West         0.830527
Other        0.912333
Name: Overall_Score, dtype: float64

REGION PERFORMANCE:
Northeast       0.784  (BETTER than national avg)
South           0.810  (BETTER than national avg)
Midwest         0.816  (BETTER than national avg)
West            0.831  (WORSE than national avg)
Other           0.912  (WORSE than national avg)
